# CMFL — Federated Learning Defense Framework

Run the CMFL paper ablation studies on Google Colab with GPU.

**Setup:** `Runtime → Change runtime type → GPU (T4 or better)`

## 1. Clone Repository & Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/RahulMadhukar/Research-Works.git /content/federated_Defence_New
%cd /content/federated_Defence_New

In [ ]:
# Install dependencies (most are pre-installed on Colab)
!pip install -q torch torchvision numpy scipy matplotlib seaborn

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → GPU")

## 2. Configuration

Adjust these settings before running experiments.

In [ ]:
# ========== CONFIGURATION ==========
# Choose which test to run:
#   'quick'       — 1 round, 25% data, natural client counts (smoke test)
#   'lightweight' — dataset-specific rounds, 50% data, 100 clients

TEST_MODE = 'lightweight'  # 'quick' or 'lightweight'

# Filter to a single dataset/attack (None = run all)
DATASET = 'femnist'           # 'femnist', 'shakespeare', 'sentiment140', or None
ATTACK  = 'gradient_scaling'  # 'gradient_scaling', 'same_value', 'back_gradient', or None

print(f"Mode: {TEST_MODE}")
print(f"Dataset: {DATASET or 'ALL'}")
print(f"Attack: {ATTACK or 'ALL'}")

## 3. (Optional) Tune Hyperparameters

Edit the learning rate and batch size directly in `run_impact_analysis.py`.

In [ ]:
# Show current hyperparameters
!grep -n 'DATASET_LR\|DATASET_BATCH_SIZE\|DATASET_ROUNDS' run_impact_analysis.py | head -5

In [ ]:
# Uncomment and modify to change hyperparameters programmatically:

# import run_impact_analysis
# run_impact_analysis.DATASET_LR['FEMNIST'] = 0.001        # default: 0.01
# run_impact_analysis.DATASET_BATCH_SIZE['FEMNIST'] = 64   # default: 32
# run_impact_analysis.DATASET_ROUNDS['FEMNIST'] = 300      # default: 600

## 4. Run Experiments

In [ ]:
# Build the command
script = 'lightweight_test.py' if TEST_MODE == 'lightweight' else 'quick_test.py'
cmd = f'python {script}'
if DATASET:
    cmd += f' -d {DATASET}'
if ATTACK:
    cmd += f' -a {ATTACK}'

print(f"Running: {cmd}")
!{cmd}

## 5. View Results

In [ ]:
# List output directories
import os
output_dir = 'Output/lightweight_ablation' if TEST_MODE == 'lightweight' else 'Output/quick_femnist_ablation'
if os.path.exists(output_dir):
    for root, dirs, files in os.walk(output_dir):
        level = root.replace(output_dir, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 3:  # don't go too deep
            for f in files:
                print(f"{indent}  {f}")
else:
    print(f"No output yet at {output_dir}")

In [ ]:
# Load and display a results JSON (update path after run)
import json, glob

json_files = glob.glob(f"{output_dir}/**/*_results.json", recursive=True)
if json_files:
    print(f"Found {len(json_files)} result files:")
    for jf in json_files[:10]:
        print(f"  {jf}")
    # Display first result
    with open(json_files[0]) as f:
        data = json.load(f)
    print(f"\n--- {json_files[0]} ---")
    print(json.dumps(data, indent=2)[:3000])
else:
    print("No result JSON files found yet.")

## 6. Save Results to Google Drive

In [ ]:
# Mount Google Drive and copy results
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/federated_Defence_New/Output /content/drive/MyDrive/CMFL_Results
print("Results saved to Google Drive: MyDrive/CMFL_Results/")